# 03 · Silver to Gold

Duas saidas:

- **`gold.ml_input`** — repassa os campos canonicos e o rotulo. E o espelho exato
  do contrato do evento do Trabalho 2: se um campo nao esta aqui, a API de outubro
  nao vai te-lo.
- **`gold.fraud_analytics`** — agregados por categoria, hora e estado, para o painel.

As duas sao agregacao e repasse sobre a tabela inteira, entao rodam como consulta
no BigQuery: o dado nem precisa sair de la.

In [ ]:
import json
import pandas as pd
from google.cloud import bigquery

In [ ]:
!pip install --quiet --upgrade google-cloud-bigquery db-dtypes

In [ ]:
try:
    from google.colab import auth
    auth.authenticate_user()
    print("Autenticado no Colab")
except ImportError:
    print("Fora do Colab: usando as credenciais do ambiente")

## Parâmetros

In [ ]:
PROJECT_ID = "fraudflow-pdm-gps"
client = bigquery.Client(project=PROJECT_ID)
print(PROJECT_ID)

## Entrada do modelo

`split_key` e o `transaction_ts` em texto no formato `'YYYY-MM-DD HH:MM:SS'`, cuja
ordem alfabetica e a ordem cronologica. Existe como alternativa pronta caso o
registro no Vertex AI recuse TIMESTAMP na entrada do modelo — ver notebook 04.

In [ ]:
SQL_ML_INPUT = f"""
-- ===========================================================================
-- 20 · gold.ml_input
--
-- Repassa os campos canônicos da Silver, mais o rótulo. NENHUMA feature é
-- calculada aqui: elas nascem inteiras dentro do TRANSFORM do CREATE MODEL,
-- para viajarem junto com o modelo quando ele for para o Vertex AI.
--
-- Esta tabela é o espelho exato do contrato do evento do Trabalho 2. Se um
-- campo não está aqui, a API de outubro não vai tê-lo.
-- ===========================================================================

CREATE OR REPLACE TABLE `{PROJECT_ID}.gold.ml_input`
PARTITION BY DATE(transaction_ts)
AS
SELECT
  -- identificador: não é feature, serve para juntar o rótulo depois da
  -- predição e para auditar um caso específico na demo
  transaction_id,

  -- ---- os dez campos do contrato ----
  transaction_ts,
  customer_dob,
  customer_lat,
  customer_long,
  merchant_lat,
  merchant_long,
  amount,
  category,
  city_pop,

  -- ---- rótulo ----
  is_fraud,

  -- Chave de corte temporal em texto, no formato 'YYYY-MM-DD HH:MM:SS'.
  -- Neste formato a ordem alfabética é a ordem cronológica, então o recorte
  -- do DATA_SPLIT_METHOD='SEQ' fica idêntico ao que sairia do TIMESTAMP.
  -- Existe como alternativa pronta caso o registro no Vertex AI recuse
  -- TIMESTAMP na entrada do modelo — ver o comentário em 30_create_model_logistic.sql.
  FORMAT_TIMESTAMP('%Y-%m-%d %H:%M:%S', transaction_ts)  AS split_key

FROM `{PROJECT_ID}.silver.transactions`;


-- Conferência: volume, taxa de fraude e janela temporal.
SELECT
  COUNT(*)                                             AS linhas,
  COUNTIF(is_fraud = 1)                                AS fraudes,
  ROUND(100 * COUNTIF(is_fraud = 1) / COUNT(*), 3)     AS pct_fraude,
  MIN(transaction_ts)                                  AS primeira,
  MAX(transaction_ts)                                  AS ultima,
  COUNTIF(amount <= 0)                                 AS valores_invalidos
FROM `{PROJECT_ID}.gold.ml_input`;
"""

In [ ]:
client.query(SQL_ML_INPUT).result()
print("gold.ml_input criada")

## O contrato, visto de perto

Uma linha no formato em que o evento vai chegar em outubro. Repare que `is_fraud`
**nao** faz parte do payload.

In [ ]:
linha = client.query(f"""
    SELECT transaction_id, transaction_ts, customer_dob,
           customer_lat, customer_long, merchant_lat, merchant_long,
           amount, category, city_pop
    FROM `{PROJECT_ID}.gold.ml_input`
    LIMIT 1
""").to_dataframe().iloc[0]

print(json.dumps({k: str(v) for k, v in linha.items()}, indent=2))

## Agregados de negócio

In [ ]:
SQL_ANALYTICS = f"""
-- ===========================================================================
-- 21 · gold.fraud_analytics
--
-- Camada de inteligência de negócio: agregados de fraude por categoria, hora
-- e estado. É o que o Looker Studio consome e o que vira slide.
--
-- Não alimenta o modelo. Aqui as agregações são livres, porque nada daqui
-- entra no TRANSFORM.
-- ===========================================================================

CREATE OR REPLACE TABLE `{PROJECT_ID}.gold.fraud_analytics` AS
SELECT
  category,
  EXTRACT(HOUR FROM transaction_ts)                        AS hour,
  customer_state,
  DATE(transaction_ts)                                     AS transaction_date,

  COUNT(*)                                                 AS transactions,
  COUNTIF(is_fraud = 1)                                    AS frauds,
  ROUND(100 * SAFE_DIVIDE(COUNTIF(is_fraud = 1), COUNT(*)), 4)
                                                           AS fraud_rate_pct,
  ROUND(SUM(amount), 2)                                    AS total_amount,
  ROUND(SUM(IF(is_fraud = 1, amount, 0)), 2)               AS fraud_amount,
  ROUND(AVG(amount), 2)                                    AS avg_ticket,
  ROUND(AVG(IF(is_fraud = 1, amount, NULL)), 2)            AS avg_ticket_fraud
FROM `{PROJECT_ID}.silver.transactions`
GROUP BY category, hour, customer_state, transaction_date;
"""

In [ ]:
client.query(SQL_ANALYTICS).result()
print("gold.fraud_analytics criada")

## Os números do slide de honestidade

Os dados vem de um simulador e as separacoes sao limpas demais para fraude real.
As metricas do modelo vao parecer otimas — melhor sermos nos a explicar por que
antes que alguem pergunte.

In [ ]:
client.query(f"""
    SELECT
      IF(hour >= 22 OR hour <= 3, 'madrugada (22h-3h)', 'resto do dia') AS periodo,
      SUM(transactions) AS transacoes,
      SUM(frauds)       AS fraudes,
      ROUND(100 * SAFE_DIVIDE(SUM(frauds), SUM(transactions)), 4) AS pct_fraude
    FROM `{PROJECT_ID}.gold.fraud_analytics`
    GROUP BY periodo
    ORDER BY pct_fraude DESC
""").to_dataframe()

In [ ]:
client.query(f"""
    SELECT
      CASE
        WHEN amount <  50   THEN 'a. ate 50'
        WHEN amount <  200  THEN 'b. 50 a 200'
        WHEN amount <  500  THEN 'c. 200 a 500'
        WHEN amount < 1000  THEN 'd. 500 a 1000'
        ELSE                     'e. acima de 1000'
      END AS faixa_valor,
      COUNT(*) AS transacoes,
      COUNTIF(is_fraud = 1) AS fraudes,
      ROUND(100 * COUNTIF(is_fraud = 1) / COUNT(*), 4) AS pct_fraude
    FROM `{PROJECT_ID}.silver.transactions`
    GROUP BY faixa_valor
    ORDER BY faixa_valor
""").to_dataframe()

## VALIDAÇÃO

In [ ]:
for nome in ['ml_input', 'fraud_analytics']:
    t = client.get_table(f"{PROJECT_ID}.gold.{nome}")
    print(f"{nome:<18} {t.num_rows:>10,} linhas   {t.num_bytes/1024**2:>7.1f} MB")

t = client.get_table(f"{PROJECT_ID}.gold.ml_input")
assert t.num_rows > 1_200_000, "ml_input com menos linhas que o esperado"
print("\nGold ok")

---

**Proximo:** `04_model_and_predict.ipynb`